# #3 Analyze gene programs

## Purpose

This notebook analyzes gene programs defined in `1_define_programs.ipynb`. It visualizes program activation across cell types and determines program rate of change along developmental trajectories.

## Setup

In [ ]:
import treedata as td
import pycea as py
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl

from devmap.config import set_theme, lineage_palette, get_paths, subtype_palette, celltype_palette
from devmap.utils import save_plot, load_data
from devmap.fate import extant_node_attribute
from devmap.plots import plot_program_lineplot
from devmap.linkage import draw_marginal_strip

set_theme()
base_path, plots_path, results_path = get_paths()

## Load data

In [ ]:
tdata = load_data("log1p_hvg",scvi = True)
cell_types = pd.read_csv(base_path / "data" / "cell_types.csv", index_col=0)

Load programs

In [ ]:
program_scores = pd.read_csv(results_path / "program_scores.csv", index_col=0)
programs = pd.read_csv(results_path / "annotated_programs.csv", index_col=0)
gene_stats = pd.read_csv(results_path / "gene_autocorrelation.csv", index_col=0)
tdata.obs = tdata.obs.merge(program_scores, left_index=True, right_index=True)

pairwise correlation

In [166]:
local_c = pd.read_csv(results_path / "local_correlations.csv", index_col=0)

## Gene selection

In [ ]:
threshold = .1
fig, ax = plt.subplots(figsize=(1.8, 1.8))
sns.histplot(data=gene_stats, x="C", bins=50, color = "lightgray")
plt.axvline(threshold, color="black", linestyle="--")
top = gene_stats.nlargest(8, "C")
# add text labels
ymax = ax.get_ylim()[1]
for i, (idx, row) in enumerate(top.iterrows()):
    ax.text(
        row["C"],
        20,
        str(idx),
        rotation=90,
        ha="right",
        fontsize=6
    )

plt.xlabel("Lineage autocorrelation")
plt.ylabel("Number of genes")
save_plot(plots_path / "gene_autocorrelation_histogram.svg", fig)

## Example programs

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi = 600)
clone = 'E9.5-R2-C3'
py.pl.tree(tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
program_genes = programs.loc["P3", "genes"].split(", ")
py.pl.annotation(tdata, keys = program_genes[:12], ax = ax, width = .1, vmax = "p100", legend = False)
save_plot(plots_path / "example_program_tree.svg", fig, rasterize = True)

In [ ]:
fig, ax = plt.subplots(figsize=(1.8, 4), dpi = 600)
clone = 'E9.5-R2-C3'
py.pl.tree(tdata, depth_key="time", tree 
           = clone, ax = ax, branch_linewidth=.5)
py.pl.annotation(tdata, keys = "P3",cmap = "PuOr_r",width = .1,vmin  = -1, vmax = 1, legend = False)
save_plot(plots_path / "example_program_score_tree.svg", fig, rasterize = True)

In [ ]:
for program in ["P3","P26","P65"]:
     fig, ax = plt.subplots(figsize=(4, 4), dpi = 600)
     sc.pl.umap(tdata, color=program, cmap = "PuOr_r", vmin = -1, vmax = 1, frameon=False, colorbar_loc = None, ax = ax)
     save_plot(plots_path / f"{program}_umap.svg", fig, rasterize = True)

## Assign colors to programs

In [ ]:
shuffled_colors = list(sc.plotting.palettes.default_102.copy())[0:len(programs)]
np.random.seed(42)
np.random.shuffle(shuffled_colors)
program_palette = dict(zip(programs.index,shuffled_colors))

## Local correlation

In [168]:
gene_assignments = programs.assign(col=programs['genes'].str.split(', ')).explode('col').reset_index(
    drop=False).rename(columns={"col": "gene"})[["gene", "program"]]

In [ ]:
# Order genes
fig, ax = plt.subplots(figsize=(4, 4), dpi = 1200)
genes = gene_assignments.gene.tolist()

# Map gene -> program -> color
gene_to_program = dict(zip(gene_assignments.gene, gene_assignments.program))
row_colors = [program_palette[gene_to_program[g]] for g in genes]

# Convert colors to array shape (n_genes, 1, 3)
row_colors_arr = np.array([sns.color_palette([c])[0] for c in row_colors]).reshape(len(genes), 1, 3)
col_colors_arr = row_colors_arr.transpose(1, 0, 2)

# Main heatmap
sns.heatmap(
    local_c.loc[genes, genes],
    cmap="RdBu_r",
    square=True,
    vmax=0.5,
    vmin=-0.5,
    ax=ax,
    cbar=True
)

# Add row color bar (left)
ax_row = ax.inset_axes([-0.03, 0, 0.02, 1])
ax_row.imshow(row_colors_arr, aspect="auto")
ax_row.axis("off")

# Add column color bar (top)
ax_col = ax.inset_axes([0, 1.02, 1, 0.02])
ax_col.imshow(col_colors_arr, aspect="auto")
ax_col.axis("off")

genes = gene_assignments.gene.tolist()
programs = gene_assignments.program.tolist()
programs_arr = np.array(programs)

# Find boundaries where program changes
change_idx = np.where(programs_arr[:-1] != programs_arr[1:])[0] + 1

# Split into contiguous blocks
blocks = np.split(np.arange(len(programs_arr)), change_idx)

# Compute tick positions (centers) and labels
tick_positions = np.array([block.mean() for block in blocks])
tick_labels = [programs_arr[block[0]] for block in blocks]

# Keep only labels that are far enough apart
min_tick_spacing = 40  # in heatmap row units; adjust as needed

keep = []
last_pos = -np.inf
for pos in tick_positions:
    if pos - last_pos >= min_tick_spacing:
        keep.append(True)
        last_pos = pos
    else:
        keep.append(False)

tick_positions_filtered = tick_positions[keep]
tick_labels_filtered = [lab for lab, k in zip(tick_labels, keep) if k]

# Apply to y-axis
ax.set_yticks(tick_positions_filtered)
ax.set_yticklabels(tick_labels_filtered, rotation=0, fontsize=6)
ax.set_xticks([])

save_plot(plots_path / "program_gene_correlation_heatmap_alternative.svg", fig, rasterize = True)


## Tree with all programs

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi = 600)
py.pl.tree(tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.annotation(tdata, keys = list(programs.index), ax = ax, width = .1, vmin = -1,
    vmax = 1, legend = False, label = False,  cmap = "PuOr_r")
save_plot(plots_path / "all_programs_tree.svg", fig, rasterize = True)

## Mean activity

In [ ]:
order = pd.read_csv("/lab/solexa_weissman/wcolgan/devmap-paper/fate/results/e95_linkage_order.csv", 
                    header=None, index_col=0).iloc[:, 0].tolist()
mean_activity = tdata.obs.query("stage == 'E9.5'").groupby(
    ["cell_type","germ_layer"], observed = True)[programs.index].mean()
exclude_programs = {"P48","P37"}

# Merge Mesoderm and Endoderm into a single group
germ_map = {
    "Mesoderm": "Mesoendo",
    "Endoderm": "Mesoendo",
    "Ectoderm": "Ectoderm",
    "Epiblast": "Ectoderm",
    "Primordial germ cell": "Ectoderm"
}

# Remap germ layers in cell_types
cell_types = cell_types.copy()
cell_types["germ_layer"] = cell_types["germ_layer"].map(germ_map).fillna(cell_types["germ_layer"])

# Max activity of each program within each merged germ layer
layer_program_max = (
    mean_activity
    .reset_index()
    .assign(germ_layer=lambda df: df["germ_layer"].map(germ_map).fillna(df["germ_layer"]))
    .groupby("germ_layer")[programs.index]
    .max()
)

germ_layers = ["Ectoderm", "Mesoendo"]
germ_layer_programs = {g: [] for g in germ_layers}
program_order = {}

threshold = 0.5
multi_assign_threshold = 0.8

for p in programs.index:
    if p in exclude_programs:
        continue
    # Germ layers where the program is strongly active
    strong_layers = layer_program_max.index[layer_program_max[p] > multi_assign_threshold].tolist()
    if len(strong_layers) > 1:
        # Assign to all germ layers with activity > 0.8
        assigned_layers = strong_layers
    else:
        # Otherwise assign only to the single germ layer with the max
        assigned_layers = [layer_program_max[p].idxmax()]
    for germ_layer in assigned_layers:
        germ_layer_programs[germ_layer].append(p)

# Make cell_type the index only
mean_activity = mean_activity.copy()
mean_activity.index = mean_activity.index.droplevel(1)

## Infer ancestral states

In [ ]:
clone = 'E9.5-R1-C1'
clone_tdata = tdata[tdata.obs["clone"] == clone].copy()
for program in programs.index:
    py.tl.ancestral_states(clone_tdata, keys = program, method = "mean", depth_key="time")

In [ ]:
fig, ax = plt.subplots(figsize=(2, 2), dpi = 600)
i = 1000
program = "P3"
py.pl.tree(clone_tdata[i:i+500], depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.nodes(clone_tdata[i:i+500], color = program, cmap = "PuOr_r", vmin = -1, vmax = 1, ax = ax, legend = False, size = 1)
py.pl.annotation(clone_tdata[i:i+500], keys = program,cmap = "PuOr_r",width = .1,vmin  = -1, vmax = 1, legend = False)

## Program rate of change

In [ ]:
def smooth_and_diff_centered(g, window=2):
    g = g.sort_values("time").copy()
    s = g["score"].rolling(window, center=True, min_periods=1).mean()

    dt = g["time"].shift(-1) - g["time"].shift(1)
    ds = s.shift(-1) - s.shift(1)

    out = ds / dt
    out.index = g.index
    return out

sampled_tdata = clone_tdata[clone_tdata.obs
    .groupby("cell_type", group_keys=False, observed = True)
    .apply(lambda x: x.sample(min(len(x), 100))).index].copy()
path_scores = extant_node_attribute(sampled_tdata,key = programs.index, depth_key="time",paths = True,
                           bins = np.arange(0, 10, .5), return_type = "dataframe")
path_scores["cell_type"] = path_scores["leaf"].map(clone_tdata.obs["cell_type"])

In [ ]:
for cell_type in ["Myotome","Eye field"]:
    # Get program activity over time
    df = path_scores.query("cell_type == @cell_type").groupby("time")[programs.index].mean().reset_index()
    germ_layer = cell_types.query("cell_type == @cell_type").germ_layer.values[0]
    df = df.melt(id_vars="time", value_vars=programs.index, var_name="program", value_name="score").query(
        "program in @germ_layer_programs[@germ_layer]")
    df = df.sort_values(["program", "time"]).copy()
    df["score_slope"] = (
        df.groupby("program", group_keys=False)
        .apply(smooth_and_diff_centered)
    )
    top_programs = (df.groupby("program")["score"].max().nlargest(5).index)
    df_top = df[df["program"].isin(top_programs)]
    df_rest = df[~df["program"].isin(top_programs)]
    # Plot program activity
    fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi = 600)
    plot_program_lineplot(df_top, df_rest, y="score", ax=ax, palette=program_palette)
    save_plot(plots_path / f"{cell_type}_program_scores.svg", fig, rasterize = True)
    # Plot program slope
    fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi = 600)
    plot_program_lineplot(df_top, df_rest, y="score_slope", ax=ax, palette=program_palette)
    save_plot(plots_path / f"{cell_type}_program_score_slopes.svg", fig, rasterize = True)

## Cell type rate of change

In [1188]:
sampled_tdata = clone_tdata[clone_tdata.obs
    .groupby("cell_subtype", group_keys=False, observed = True)
    .apply(lambda x: x.sample(min(len(x), 100))).index].copy()
subtype_scores = extant_node_attribute(sampled_tdata,key = programs.index, depth_key="time",paths = True,
                           bins = np.arange(0, 10, .5), return_type = "dataframe")
subtype_scores["cell_subtype"] = subtype_scores["leaf"].map(clone_tdata.obs["cell_subtype"])

/tmp/ipykernel_2130132/84281161.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 100))).index].copy()


P33

In [ ]:
program = "P33"
df = subtype_scores.groupby(["cell_subtype", "time"])[program].mean().reset_index()
top_subtypes = (df.groupby("cell_subtype")[program].max().nlargest(5).index)
df_top = df[df["cell_subtype"].isin(top_subtypes)]
df_rest = df[~df["cell_subtype"].isin(top_subtypes)]

fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi = 600)
plot_program_lineplot(df_top, df_rest, y=program, ax=ax,  palette = subtype_palette, hue = "cell_subtype")
save_plot(plots_path / f"{program}_subtype_scores.svg", fig, rasterize = True)

P44

In [ ]:
program = "P44"
df = path_scores.groupby(["cell_type", "time"])[program].mean().reset_index()
top_subtypes = (df.groupby("cell_type")[program].max().nlargest(5).index)
df_top = df[df["cell_type"].isin(top_subtypes)]
df_rest = df[~df["cell_type"].isin(top_subtypes)]

fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi = 600)
plot_program_lineplot(df_top, df_rest, y=program, ax=ax,  palette = celltype_palette, hue = "cell_type")
save_plot(plots_path / f"{program}_subtype_scores.svg", fig, rasterize = True)

## Rate of change for all programs

In [ ]:
programs["label"] = programs["name"] + " (" + programs.index + ")"

program_scores_long = (
    path_scores
    .groupby(["time", "cell_type"])[programs.index]
    .mean()
    .reset_index()
    .melt(
        id_vars=["time", "cell_type"],
        value_vars=programs.index,
        var_name="program",
        value_name="score",
    )
)

max_score_by_program_cell_type = (
    program_scores_long
    .groupby(["program", "cell_type"])["score"]
    .max()
)

active_program_cell_type_pairs = []

for program, program_max_scores in max_score_by_program_cell_type.groupby(level="program"):
    pairs_above_threshold = program_max_scores[program_max_scores > 0.5]

    if len(pairs_above_threshold):
        active_program_cell_type_pairs.extend(pairs_above_threshold.index.tolist())
    else:
        active_program_cell_type_pairs.append(program_max_scores.idxmax())

active_program_scores = (
    program_scores_long
    .set_index(["program", "cell_type"])
    .loc[active_program_cell_type_pairs]
    .reset_index()
)

active_program_scores["score_slope"] = (
    active_program_scores
    .groupby(["program", "cell_type"], group_keys=False)
    .apply(lambda group: smooth_and_diff_centered(group, window=2))
)

active_program_scores["time"] = active_program_scores["time"] + 0.5

program_slope_matrix = (
    active_program_scores
    .groupby(["program", "time"])["score_slope"]
    .mean()
    .unstack("time")
)

normalized_slope_matrix = program_slope_matrix.div(program_slope_matrix.abs().max(axis=1), axis=0)

## Plot rate of change heatmap

In [ ]:
cell_type_unique = cell_types.groupby("cell_type").agg({
    "lineage": lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    "germ_layer": lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    "id": lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    "type_color": lambda x: x.mode()[0] if not x.mode().empty else np.nan
})
germ_layer_order = {}

In [ ]:
for germ_layer in ["Ectoderm", "Mesoendo"]:
    
    mat = mean_activity.loc[program_order[germ_layer], germ_layer_programs[germ_layer]]
    g = sns.clustermap(mat)
    type_order = mat.index[g.dendrogram_row.reordered_ind].values.tolist()
    program_order = mat.columns[g.dendrogram_col.reordered_ind].values.tolist()
    germ_layer_order[germ_layer] = program_order
    plt.close(g.fig)

    fig, axes = plt.subplots(
        2, 2,
        figsize=(6.9, 5.5),
        dpi=600,
        gridspec_kw={"width_ratios": [2.5, 1], "height_ratios": [.05, .95]}  # make left plot wider
    )

    score_ax = axes[1, 0]
    rate_ax = axes[1, 1]
    ax_col = axes[0, 0]
    # Activity heatmap
    sns.heatmap(
        mean_activity.loc[type_order, program_order].T,
        ax = score_ax,
        square=False,
        vmax=1,
        vmin=-1,
        cmap="PuOr_r",
        xticklabels=type_order,
        yticklabels=programs.loc[program_order, "label"],
        cbar=False
    )

    # Rate of change heatmap
    sns.heatmap(
        normalized_slope_matrix.loc[program_order],  # ensure same order as left heatmap
        cmap="RdBu_r",
        center=0,
        vmax=1,
        vmin=-1,
        ax=rate_ax,
        cbar = False
    )

    draw_marginal_strip(ax_col, cell_type_unique["lineage"], lineage_palette, ordered_index=type_order, orientation="col")

    # Remove y ticks and labels from right plot
    rate_ax.set_yticks([])
    rate_ax.set_yticklabels([])
    score_ax.tick_params(axis='y', labelsize=6)
    score_ax.tick_params(axis='x', labelsize=6)
    rate_ax.tick_params(axis='x', labelsize=8)
    desired = [1.5, 3.5, 5.5, 7.5, 9.5]
    tick_idx = [normalized_slope_matrix.columns.get_loc(x) for x in desired]
    rate_ax.set_xticks([i + 0.5 for i in tick_idx])
    rate_ax.set_xticklabels(desired)
    rate_ax.set_ylabel("")
    plt.tight_layout()
    save_plot(plots_path / f"{germ_layer.lower()}_programs_heatmap.svg", fig)


## Activity across timpoints

In [ ]:
cmap = sns.color_palette("PuOr_r", as_cmap=True)
cmap.set_bad(color="lightgray")
mean_by_stage = tdata.obs.groupby(["stage","cell_type"])[programs.index].mean().reset_index()
stages = mean_by_stage["stage"].unique()

for germ_layer in ["Ectoderm", "Mesoendo"]:
    program_order = germ_layer_order[germ_layer]
    use_types = cell_types.query("germ_layer == @germ_layer").index.tolist()
    fig, axes = plt.subplots(1, len(use_types), figsize=(.16*len(use_types), 5.5), squeeze=False, dpi = 600)
    for i, ct in enumerate(use_types):
        ax = axes[0, i]
        df_ct = mean_by_stage[mean_by_stage["cell_type"] == ct]
        heatmap_data = (
            df_ct.set_index("stage")[program_order]
            .reindex(stages)
            .T
        )
        sns.heatmap(
            heatmap_data,
            ax=ax,
            cmap=cmap,
            mask=heatmap_data.isna(),
            cbar=False,
            vmax = 1,
            vmin = -1,
            yticklabels = programs.loc[heatmap_data.index, "label"]
        )
        ax.set_xlabel(f"{ct}", fontsize=6, rotation = 90)
        ax.set_xticks([])
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=6)
        # remove y ticks
        if i > 0:
            ax.set_yticks([])
            ax.set_ylabel("")

    plt.tight_layout()
    fig.subplots_adjust(wspace=0.2)
    #plt.show()
    save_plot(plots_path / f"ectoderm_programs_by_stage_heatmap.svg", fig, rasterize = True)



## Differentiation rate

In [ ]:
py.tl.tree_neighbors(tdata, depth_key="time", max_dist = 3, update = False)
X_scvi = tdata.obsm["X_scvi"]
conn = tdata.obsp["tree_connectivities"]

rows, cols = conn.nonzero()
dists = np.linalg.norm(X_scvi[rows] - X_scvi[cols], axis=1)

sum_dists = np.bincount(rows, weights=dists, minlength=tdata.n_obs)
n_neighbors = np.bincount(rows, minlength=tdata.n_obs)

mean_neighbor_pca_dist = np.divide(
    sum_dists,
    n_neighbors,
    out=np.full(tdata.n_obs, np.nan),
    where=n_neighbors > 0
)
tdata.obs["mean_neighbor_scvi_dist"] = mean_neighbor_pca_dist

On UMAP

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 3), dpi = 600)
sc.pl.umap(tdata[tdata.obs.query("mean_neighbor_scvi_dist.notnull()").index], frameon=False,title = "",
            color="mean_neighbor_scvi_dist", cmap = "RdPu", vmax = 12,vmin = 6, sort_order=False, ax = ax)
save_plot(plots_path / "neighbor_scvi_distance_umap.svg", fig, rasterize = True)

Mean per cell type

In [ ]:
df = tdata.obs.query("mean_neighbor_scvi_dist.notnull()").groupby(
    "cell_type", observed=True)["mean_neighbor_scvi_dist"].median().sort_values(ascending=False).reset_index()
df["rank"] = df["mean_neighbor_scvi_dist"].rank(ascending=False)
df = df.sort_values("mean_neighbor_scvi_dist", ascending=True)
fig, ax = plt.subplots(figsize=(1.6, 1.6), dpi = 600)
norm = mpl.colors.Normalize(vmin=6, vmax=12)
cmap = plt.get_cmap("RdPu")
colors = cmap(norm(df["mean_neighbor_scvi_dist"].values))
sns.scatterplot(data=df, x="rank", y="mean_neighbor_scvi_dist", color = colors, edgecolor ="black", ax = ax, s = 10)
plt.xticks([0,20,40,60,80,100]);
save_plot(plots_path / "neighbor_scvi_distance_rank.svg", fig, rasterize = False)